# LangGraph agent using Qwen3.5-0.8B


In [ ]:
!pip install -q langchain langgraph langchain-community "langchain[huggingface]" sentence-transformers langchain-text-splitters

In [ ]:
import json
import math
import re
from pathlib import Path
from uuid import uuid4

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from langchain.agents import create_agent
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError

## Model wrapper

Renders the chat with the tool schemas, generates, and parses Qwen3.5's XML tool calls back into LangChain `tool_calls`.

In [ ]:
MODEL_DIR = "./models/Qwen3.5-0.8B"
MODEL_ID = "Qwen/Qwen3.5-0.8B"

_TOOL_RE = re.compile(r"<tool_call>\s*<function=([^>\s]+)>(.*?)</function>\s*</tool_call>", re.S)
_PARAM_RE = re.compile(r"<parameter=([^>\s]+)>\n?(.*?)\n?</parameter>", re.S)
_ROLES = {"system": "system", "human": "user", "ai": "assistant", "tool": "tool"}


def parse_tool_calls(text: str) -> tuple[str, list[dict]]:
    """Split model output into (plain text, tool_calls) for Qwen3.5's XML tool format."""
    calls = []
    for name, body in _TOOL_RE.findall(text):
        args = {}
        for k, v in _PARAM_RE.findall(body):
            v = v.strip()
            try:
                args[k] = json.loads(v)  # numbers, bools; falls back to raw string
            except json.JSONDecodeError:
                args[k] = v
        calls.append({"name": name, "args": args, "id": uuid4().hex[:8], "type": "tool_call"})
    content = _TOOL_RE.sub("", text).split("</think>")[-1].replace("<tool_call>", "").strip()
    return content, calls


def _to_dicts(messages) -> list[dict]:
    """LangChain messages -> Qwen's chat template."""
    out = []
    for m in messages:
        d = {"role": _ROLES[m.type], "content": m.content if isinstance(m.content, str) else str(m.content)}
        if m.type == "ai" and m.tool_calls:
            d["tool_calls"] = [{"function": {"name": c["name"], "arguments": c["args"]}} for c in m.tool_calls]
        out.append(d)
    return out


class ChatQwen(BaseChatModel):
    """Local Qwen3.5 as a LangChain chat model."""

    def __init__(self):
        super().__init__()
        src = MODEL_DIR if Path(MODEL_DIR).exists() else MODEL_ID
        self._tok = AutoTokenizer.from_pretrained(src)
        self._model = AutoModelForCausalLM.from_pretrained(
            src, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="cuda" if torch.cuda.is_available() else "cpu")
        # Define <|im_end|> explicitly;
        self._stop_ids = [self._tok.convert_tokens_to_ids("<|im_end|>"), self._tok.eos_token_id]

    @property
    def _llm_type(self) -> str:
        return "qwen3.5-local"

    def bind_tools(self, tools, **kwargs):
        return self.bind(tools=[convert_to_openai_tool(t) for t in tools], **kwargs)

    def _generate(self, messages, tools=None, **kwargs) -> ChatResult:
        prompt = self._tok.apply_chat_template(
                _to_dicts(messages), 
                tools=tools or [], 
                add_generation_prompt=True,
                tokenize=False, 
                enable_thinking=False
            )
        inputs = self._tok(prompt, return_tensors="pt").to(self._model.device)
        with torch.inference_mode():
            out = self._model.generate(**inputs, max_new_tokens=1024, do_sample=False,
                                       eos_token_id=self._stop_ids)
        text = self._tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        content, calls = parse_tool_calls(text)
        # One call per turn: LangChain runs a turn's tool calls in parallel, which breaks
        # order-dependent (stateful) tools. The model sees each result before the next call.
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=content, tool_calls=calls[:1]))])

In [ ]:
llm = ChatQwen()

## Tools

Mock drone-fleet tools: they return canned strings and don't control anything.

In [ ]:
# Notebook-only visualisation. In deployment, delete this cell, the `show_path(...)` call in
# plan_path, and nothing else needs to change.
from IPython.display import display
from PIL import Image, ImageDraw


def draw_path(xy: list[tuple[float, float]], radius: float, size: int = 600, margin: int = 40) -> Image.Image:
    """Render waypoints (metres east/north of the centre) over the search circle."""
    img = Image.new("RGB", (size, size), "white")
    d = ImageDraw.Draw(img)
    scale = (size / 2 - margin) / radius
    to_px = lambda x, y: (size / 2 + x * scale, size / 2 - y * scale)  # north is up

    c, r = size / 2, radius * scale
    d.ellipse([c - r, c - r, c + r, c + r], outline="gray", width=2)
    d.line([to_px(*p) for p in xy], fill="royalblue", width=2)
    for i, p in enumerate(xy, 1):
        px, py = to_px(*p)
        colour = "green" if i == 1 else "red" if i == len(xy) else "royalblue"
        d.ellipse([px - 5, py - 5, px + 5, py + 5], fill=colour)
        d.text((px + 7, py - 12), str(i), fill="black")
    d.text((10, 10), f"radius {radius:g} m, {len(xy)} waypoints (green=start, red=end)", fill="black")
    return img


def show_path(xy: list[tuple[float, float]], radius: float) -> None:
    display(draw_path(xy, radius))

In [ ]:
M_PER_DEG_LAT = 111_320.0
CRUISE_MPS = 5.0
VALID_PATTERNS = ("lawnmower", "spiral")


def _lawnmower_xy(radius: float, spacing: float) -> list[tuple[float, float]]:
    """Boustrophedon: east-west sweeps clipped to the circle, alternating direction."""
    n = int(radius // spacing)
    pts, west_to_east = [], True
    for i in range(-n, n + 1):
        y = i * spacing
        half = math.sqrt(max(radius**2 - y**2, 0.0))
        line = [(-half, y), (half, y)]
        pts += line if west_to_east else line[::-1]
        west_to_east = not west_to_east
    return pts


def _spiral_xy(radius: float, spacing: float, points_per_turn: int = 6) -> list[tuple[float, float]]:
    """Archimedean spiral from the centre outwards; rings are `spacing` apart."""
    pts = [(0.0, 0.0)]
    step = 2 * math.pi / points_per_turn
    theta = step
    while True:
        r = min(spacing * theta / (2 * math.pi), radius)
        pts.append((r * math.cos(theta), r * math.sin(theta)))
        if r >= radius:
            return pts
        theta += step


@tool
def plan_path(latitude: float, longitude: float, radius_m: float, pattern: str = "lawnmower") -> str:
    """Plan a search path that covers a circular region and return its waypoints.

    Use this before navigate: it only plans, it does not move any drone.
    Call it once per region, then send the returned waypoints to a drone
    with navigate, in order.

    Args:
        latitude: Centre of the region in decimal degrees (-90 to 90).
        longitude: Centre of the region in decimal degrees (-180 to 180).
        radius_m: Radius of the region in metres (typically 50 to 2000).
        pattern: Sweep style: 'lawnmower' (parallel lines, best for open
            ground) or 'spiral' (outward from the centre, best when the
            last known position is near the centre).
    """
    pattern = pattern.strip().lower()
    if pattern not in VALID_PATTERNS:
        return f"error: unknown pattern '{pattern}', use one of {', '.join(VALID_PATTERNS)}"
    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        return "error: latitude must be in [-90, 90] and longitude in [-180, 180]"
    if radius_m <= 0:
        return "error: radius_m must be positive"

    spacing = radius_m / 3  # sweep width between adjacent lines / rings
    xy = _lawnmower_xy(radius_m, spacing) if pattern == "lawnmower" else _spiral_xy(radius_m, spacing)

    m_per_deg_lon = M_PER_DEG_LAT * max(math.cos(math.radians(latitude)), 1e-6)
    waypoints = [(round(latitude + y / M_PER_DEG_LAT, 6), round(longitude + x / m_per_deg_lon, 6))
                 for x, y in xy]
    length = sum(math.dist(a, b) for a, b in zip(xy, xy[1:]))
    minutes = length / CRUISE_MPS / 60

    show_path(xy, radius_m)  # notebook-only; remove in deployment

    lines = [f"{i}. ({la}, {lo})" for i, (la, lo) in enumerate(waypoints, 1)]
    return (
        f"path planned ({pattern}) over ({latitude}, {longitude}) radius {radius_m} m: "
        f"{len(waypoints)} waypoints, total length {length / 1000:.2f} km, "
        f"est. flight time {minutes:.0f} min\n" + "\n".join(lines)
    )


@tool
def list_drones() -> str:
    """List every drone in the fleet with its id, state, position and battery.

    Call this first to discover which drone ids exist and which are free
    (state 'idle') before giving any drone work. Drones with battery below
    25% should not be sent on new searches.
    """
    return (
        "alpha-1: idle, at base, battery 92%\n"
        "alpha-2: searching, lat 1.3521 lon 103.8198, battery 61%\n"
        "bravo-1: returning to base, battery 18%"
    )


@tool
def navigate(drone_id: str, latitude: float, longitude: float, altitude_m: float = 60.0) -> str:
    """Command one drone to fly to a single point. Call it repeatedly, once
    per waypoint, to fly a path from plan_path. The drone must exist
    (see list_drones) and have enough battery to get there and back.

    Args:
        drone_id: Drone identifier as returned by list_drones, e.g. 'alpha-1'.
        latitude: Destination in decimal degrees (-90 to 90).
        longitude: Destination in decimal degrees (-180 to 180).
        altitude_m: Flight altitude in metres above ground (10 to 120).
    """
    return f"ok: {drone_id} flying to ({latitude}, {longitude}) at {altitude_m} m, ETA 4 min"


def list_tools() -> list:
    return [plan_path, list_drones, navigate, search_docs]

## Documents (RAG)

Indexes the files in `docs/` once, in memory, and exposes them as the `search_docs` tool. The agent decides
when to call it. Run this cell before the Agent cell.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCS_DIR = Path("docs")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

files = [p for p in sorted(DOCS_DIR.glob("*")) if p.suffix in (".md", ".txt")]
chunks = splitter.create_documents(
    [p.read_text(encoding="utf-8") for p in files],
    metadatas=[{"source": p.name} for p in files],
)
store = InMemoryVectorStore(embeddings)
if chunks:
    store.add_documents(chunks)
print(f"indexed {len(chunks)} chunks from {len(files)} files in {DOCS_DIR.resolve()}")


@tool
def search_docs(query: str) -> str:
    """Search the operations documents for procedures, limits and rules.

    Use this when a request depends on documented policy, e.g. battery
    minimums, altitude limits or flight rules. It only reads documents;
    it does not move any drone.

    Args:
        query: What to look for, in a few words, e.g. 'minimum battery for a search'.
    """
    hits = store.similarity_search(query, k=3)
    if not hits:
        return "no matching documents found"
    return "\n\n".join(f"[{h.metadata.get('source', '?')}]\n{h.page_content}" for h in hits)

## Agent

In [ ]:
MAX_STEPS = 12

SYSTEM = """You are a helpful assistant with access to tools. Fulfil the user's request by calling tools when needed.
Rules:
- Call one tool at a time, in order. Never invent tools or arguments.
- If a tool returns an error, read it and adapt, or tell the user why you cannot do it.
- For questions about procedures, limits or rules, call search_docs and answer only from what it returns.
- When you have the answer, reply briefly. Do not call more tools."""

agent = create_agent(model=llm,
                     tools=list_tools(),
                     system_prompt=SYSTEM,
                     checkpointer=InMemorySaver())

## Chat

`ask()` sends one prompt and streams the replies and tool calls. Turns that share a `thread_id` share
conversation memory; pass a new `thread_id` to start a fresh conversation.

In [ ]:
def ask(text: str, thread_id: str = "1") -> None:
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 2 * MAX_STEPS + 1}
    try:
        stream = agent.stream_events(
            {"messages": [{"role": "user", "content": text}]},
            config=config,
            version="v3",
        )
        for kind, item in stream.interleave("messages", "tool_calls"):
            if kind == "messages":
                for token in item.text:
                    print(token, end="", flush=True)
            elif kind == "tool_calls":
                print(f"\nTool call: {item.tool_name}({item.input})")
                for delta in item.output_deltas:
                    print(delta, end="", flush=True)
                print(f"\nTool result: {item.output}")
        print()
    except GraphRecursionError:
        print("\nAgent> Step limit reached.")
    except Exception as e:
        print(f"\nAgent> Error: {type(e).__name__}: {e}")

In [ ]:
ask("Which drones are free right now?")

In [ ]:
ask("Plan a spiral search with a 300 m radius around 1.3521, 103.8198 and send the free drone to its centre.")

Interactive loop, like the original script. Submit an empty prompt to stop.

In [ ]:
while text := input("Your prompt: ").strip():
    ask(text)